# Historic Algorithm Performance Analysis

Compare KPIs across two date ranges — **before** and **after** algorithm adoption.

**Prerequisites:**
1. Run `scripts/analysis/historic_fetch.py` for each period to collect API snapshots.
2. Run `scripts/analysis/historic_evaluate.py` for each period to compute `daily_kpis.csv`.

Edit the **Configuration** cell only.

In [1]:
# ── Configuration ──────────────────────────────────────────────────────────────
from pathlib import Path

_PROJECT_ROOT = Path("../..").resolve()
DATA_DIR = _PROJECT_ROOT / "data" / "historic_analysis"

PERIOD_BEFORE = "before"   # folder name used in --period-name for the pre-adoption period
PERIOD_AFTER  = "after"    # folder name used in --period-name for the post-adoption period

# Optional: vertical reference line marking the adoption date (YYYY-MM-DD or None)
ADOPTION_DATE = None  # e.g. "2025-04-01"

# Color palette
COLOR_BEFORE = "#EF553B"
COLOR_AFTER  = "#636EFA"

print(f"Project root: {_PROJECT_ROOT}")
print(f"Data dir:     {DATA_DIR}")

Project root: /Users/jbeta/Documents/AlfredProject/AlfredDEV
Data dir:     /Users/jbeta/Documents/AlfredProject/AlfredDEV/data/historic_analysis


In [2]:
# ── Load KPI CSVs ──────────────────────────────────────────────────────────────
import pandas as pd

def _load_period(period_name: str) -> pd.DataFrame:
    csv_path = DATA_DIR / period_name / "daily_kpis.csv"
    if not csv_path.exists():
        raise FileNotFoundError(
            f"KPI file not found: {csv_path}\n"
            f"Run historic_evaluate.py --period-name {period_name} first."
        )
    df = pd.read_csv(csv_path, parse_dates=["date"])
    df["period"] = period_name
    # Derived normalized distance metrics
    df["move_km_per_vt_labor"] = (
        df["total_driver_move_distance_km"]
        / df["vt_labors_total"].replace(0, float("nan"))
    )
    df["move_pct_of_total_distance"] = (
        df["total_driver_move_distance_km"]
        / (df["total_driver_move_distance_km"] + df["total_labor_distance_km"]).replace(0, float("nan"))
        * 100
    )
    return df

df_before = _load_period(PERIOD_BEFORE)
df_after  = _load_period(PERIOD_AFTER)
df_all    = pd.concat([df_before, df_after], ignore_index=True).sort_values("date").reset_index(drop=True)

print(f"Before period: {df_before['date'].min().date()} → {df_before['date'].max().date()} ({len(df_before)} days)")
print(f"After  period: {df_after['date'].min().date()} → {df_after['date'].max().date()} ({len(df_after)} days)")
print(f"Total rows: {len(df_all)}")

Before period: 2026-04-10 → 2026-04-21 (8 days)
After  period: 2026-05-11 → 2026-05-21 (8 days)
Total rows: 16


In [3]:
# ── Data Availability Check ────────────────────────────────────────────────────
def _availability_report(df: pd.DataFrame, period_name: str) -> pd.DataFrame:
    total = len(df)
    zero_services = int((df["services_total"] == 0).sum())
    skipped = int(df.get("skip_reason", pd.Series(dtype=object)).notna().sum()) if "skip_reason" in df.columns else 0
    valid = total - zero_services
    return pd.DataFrame([{
        "period": period_name,
        "total_days": total,
        "days_with_services": valid,
        "days_zero_services": zero_services,
        "days_skipped": skipped,
        "date_range": f"{df['date'].min().date()} → {df['date'].max().date()}",
    }])

availability = pd.concat([
    _availability_report(df_before, PERIOD_BEFORE),
    _availability_report(df_after,  PERIOD_AFTER),
], ignore_index=True).set_index("period")

display(availability)

# Flag days with zero services
for period_name, df in [(PERIOD_BEFORE, df_before), (PERIOD_AFTER, df_after)]:
    zero_days = df.loc[df["services_total"] == 0, "date"].dt.date.tolist()
    if zero_days:
        print(f"  [{period_name}] zero-service days ({len(zero_days)}): {zero_days}")

,total_days,days_with_services,days_zero_services,days_skipped,date_range
period,,,,,
before,8,8,0,0,2026-04-10 → 2026-04-21
after,8,8,0,0,2026-05-11 → 2026-05-21


In [4]:
# ── Statistical Summary ────────────────────────────────────────────────────────
_kpi_cols = [
    "services_total", "vt_labors_total", "drivers_used",
    "total_labor_distance_km", "total_driver_move_distance_km",
    "move_km_per_vt_labor", "move_pct_of_total_distance",
    "utilization_without_moves_pct", "utilization_with_moves_pct", "driver_move_utilization_pct",
    "late_services_pct", "normalized_tardiness_pct", "total_lateness_min",
    "preassigned_infeasible",
]

_desc_before = df_before[_kpi_cols].describe().T[["mean", "std", "min", "50%", "max"]]
_desc_after  = df_after[_kpi_cols].describe().T[["mean", "std", "min", "50%", "max"]]
_desc_before.columns = [f"{PERIOD_BEFORE}_{c}" for c in _desc_before.columns]
_desc_after.columns  = [f"{PERIOD_AFTER}_{c}"  for c in _desc_after.columns]

stats_table = _desc_before.join(_desc_after).round(2)
display(stats_table)

,before_mean,before_std,before_min,before_50%,before_max,after_mean,after_std,after_min,after_50%,after_max
services_total,42.38,4.21,34.00,44.00,47.00,42.62,7.01,33.00,45.00,52.00
vt_labors_total,48.88,5.22,40.00,48.50,56.00,48.50,7.71,37.00,49.50,59.00
drivers_used,13.12,0.35,13.00,13.00,14.00,13.62,0.52,13.00,14.00,14.00
total_labor_distance_km,652.59,726.69,241.06,410.84,2437.41,351.83,52.31,241.43,372.82,411.43
total_driver_move_distance_km,447.44,69.03,364.35,443.10,542.75,446.01,29.85,408.93,440.16,502.50
move_km_per_vt_labor,9.14,0.85,7.44,9.28,10.05,9.36,1.31,7.74,9.55,11.41
move_pct_of_total_distance,48.33,13.12,18.21,50.40,60.32,56.09,3.60,51.51,55.25,63.93
utilization_without_moves_pct,35.40,12.59,24.25,32.37,64.57,29.72,4.94,24.84,28.47,37.75
utilization_with_moves_pct,45.35,13.12,32.44,42.50,74.88,39.06,5.58,34.22,36.87,47.45
driver_move_utilization_pct,9.95,1.20,8.19,9.91,12.02,9.34,0.99,7.86,9.27,10.65


In [5]:
# ── Before / After Comparison Table ───────────────────────────────────────────
import numpy as np

_kpi_labels = {
    "services_total":                "Services per day",
    "vt_labors_total":               "VT labors per day",
    "drivers_used":                  "Drivers used",
    "total_labor_distance_km":       "Total labor distance (km)",
    "total_driver_move_distance_km": "Total driver move distance (km)",
    "move_km_per_vt_labor":          "Driver move distance per VT labor (km/labor)",
    "move_pct_of_total_distance":    "Driver move as % of total distance",
    "utilization_without_moves_pct": "Utilization excl. moves (%)",
    "utilization_with_moves_pct":    "Utilization incl. moves (%)",
    "driver_move_utilization_pct":   "Driver move utilization (%)",
    "late_services_pct":             "Late services (%)",
    "normalized_tardiness_pct":      "Normalized tardiness (%)",
    "total_lateness_min":            "Total lateness (min)",
    "preassigned_infeasible":        "Infeasible reconstructions",
    "excluded_labors_count":         "Excluded outlier labors",
}

# Only include rows where both periods have data
_df_b_valid = df_before[df_before["services_total"] > 0]
_df_a_valid = df_after[df_after["services_total"] > 0]

rows_comparison = []
for col, label in _kpi_labels.items():
    col_b = df_before.get(col) if hasattr(df_before, "get") else (df_before[col] if col in df_before.columns else None)
    col_a = df_after.get(col) if hasattr(df_after, "get") else (df_after[col] if col in df_after.columns else None)
    if col_b is None or col not in df_before.columns:
        continue
    mean_b = _df_b_valid[col].mean() if col in _df_b_valid.columns else float("nan")
    mean_a = _df_a_valid[col].mean() if col in _df_a_valid.columns else float("nan")
    delta = mean_a - mean_b
    delta_pct = (delta / mean_b * 100) if mean_b and not np.isnan(mean_b) else np.nan
    rows_comparison.append({
        "KPI": label,
        f"{PERIOD_BEFORE} (mean)": round(mean_b, 2),
        f"{PERIOD_AFTER} (mean)": round(mean_a, 2),
        "Δ (absolute)": round(delta, 2),
        "Δ (%)": round(delta_pct, 1) if not np.isnan(delta_pct) else "—",
    })

comparison_df = pd.DataFrame(rows_comparison).set_index("KPI")
display(comparison_df)

,before (mean),after (mean),Δ (absolute),Δ (%)
KPI,,,,
Services per day,42.38,42.62,0.25,0.6
VT labors per day,48.88,48.50,-0.38,-0.8
Drivers used,13.12,13.62,0.50,3.8
Total labor distance (km),652.59,351.83,-300.76,-46.1
Total driver move distance (km),447.44,446.01,-1.42,-0.3
Driver move distance per VT labor (km/labor),9.14,9.36,0.22,2.4
Driver move as % of total distance,48.33,56.09,7.77,16.1
Utilization excl. moves (%),35.40,29.72,-5.68,-16.1
Utilization incl. moves (%),45.35,39.06,-6.29,-13.9


In [6]:
# ── Chart: Services & Labors Per Day ──────────────────────────────────────────
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def _add_adoption_vline(fig, row=1, col=1):
    if ADOPTION_DATE:
        fig.add_vline(
            x=ADOPTION_DATE, line_dash="dash", line_color="gray",
            annotation_text="Adoption", annotation_position="top right",
            row=row, col=col,
        )

fig_svcs = go.Figure()
for period_name, df, color in [
    (PERIOD_BEFORE, df_before, COLOR_BEFORE),
    (PERIOD_AFTER,  df_after,  COLOR_AFTER),
]:
    fig_svcs.add_trace(go.Scatter(
        x=df["date"], y=df["services_total"],
        mode="lines+markers", name=f"{period_name} — services",
        line=dict(color=color), marker=dict(size=4),
    ))
    fig_svcs.add_trace(go.Scatter(
        x=df["date"], y=df["vt_labors_total"],
        mode="lines", name=f"{period_name} — VT labors",
        line=dict(color=color, dash="dot"),
    ))

if ADOPTION_DATE:
    fig_svcs.add_vline(x=ADOPTION_DATE, line_dash="dash", line_color="gray",
                       annotation_text="Adoption", annotation_position="top right")

fig_svcs.update_layout(
    title="Services & VT Labors per Day",
    xaxis_title="Date", yaxis_title="Count",
    legend=dict(orientation="h", y=-0.2),
    template="plotly_white",
)
fig_svcs.show()

In [7]:
# ── Chart: Driver Utilization ──────────────────────────────────────────────────
fig_util = go.Figure()
for period_name, df, color in [
    (PERIOD_BEFORE, df_before, COLOR_BEFORE),
    (PERIOD_AFTER,  df_after,  COLOR_AFTER),
]:
    fig_util.add_trace(go.Scatter(
        x=df["date"], y=df["utilization_with_moves_pct"],
        mode="lines+markers", name=f"{period_name} — incl. moves",
        line=dict(color=color), marker=dict(size=4),
    ))
    fig_util.add_trace(go.Scatter(
        x=df["date"], y=df["utilization_without_moves_pct"],
        mode="lines", name=f"{period_name} — excl. moves",
        line=dict(color=color, dash="dot"),
    ))

if ADOPTION_DATE:
    fig_util.add_vline(x=ADOPTION_DATE, line_dash="dash", line_color="gray",
                       annotation_text="Adoption", annotation_position="top right")

fig_util.update_layout(
    title="Driver Utilization (%) Over Time",
    xaxis_title="Date", yaxis_title="Utilization (%)",
    legend=dict(orientation="h", y=-0.2),
    template="plotly_white",
)
fig_util.show()

In [8]:
# ── Chart: Punctuality ────────────────────────────────────────────────────────
fig_punct = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    subplot_titles=("Late Services (%)", "Normalized Tardiness (%)"),
    vertical_spacing=0.12,
)
for period_name, df, color in [
    (PERIOD_BEFORE, df_before, COLOR_BEFORE),
    (PERIOD_AFTER,  df_after,  COLOR_AFTER),
]:
    fig_punct.add_trace(
        go.Scatter(x=df["date"], y=df["late_services_pct"],
                   mode="lines+markers", name=period_name,
                   line=dict(color=color), marker=dict(size=4), legendgroup=period_name),
        row=1, col=1,
    )
    fig_punct.add_trace(
        go.Scatter(x=df["date"], y=df["normalized_tardiness_pct"],
                   mode="lines+markers", name=period_name,
                   line=dict(color=color), marker=dict(size=4),
                   legendgroup=period_name, showlegend=False),
        row=2, col=1,
    )

if ADOPTION_DATE:
    for r in [1, 2]:
        fig_punct.add_vline(x=ADOPTION_DATE, line_dash="dash", line_color="gray", row=r, col=1)

fig_punct.update_layout(
    title="Punctuality Over Time",
    height=500,
    legend=dict(orientation="h", y=-0.15),
    template="plotly_white",
)
fig_punct.show()

In [8]:
# ── Chart: Absolute Distance KPIs ─────────────────────────────────────────────
fig_dist = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    subplot_titles=("Total Labor Distance (km)", "Total Driver Move Distance (km)"),
    vertical_spacing=0.12,
)
for period_name, df, color in [
    (PERIOD_BEFORE, df_before, COLOR_BEFORE),
    (PERIOD_AFTER,  df_after,  COLOR_AFTER),
]:
    fig_dist.add_trace(
        go.Scatter(x=df["date"], y=df["total_labor_distance_km"],
                   mode="lines+markers", name=period_name,
                   line=dict(color=color), marker=dict(size=4), legendgroup=period_name),
        row=1, col=1,
    )
    fig_dist.add_trace(
        go.Scatter(x=df["date"], y=df["total_driver_move_distance_km"],
                   mode="lines+markers", name=period_name,
                   line=dict(color=color), marker=dict(size=4),
                   legendgroup=period_name, showlegend=False),
        row=2, col=1,
    )

if ADOPTION_DATE:
    for r in [1, 2]:
        fig_dist.add_vline(x=ADOPTION_DATE, line_dash="dash", line_color="gray", row=r, col=1)

fig_dist.update_layout(
    title="Absolute Distance KPIs Over Time",
    height=500,
    legend=dict(orientation="h", y=-0.15),
    template="plotly_white",
)
fig_dist.show()

In [10]:
# ── Chart: Normalized Distance KPIs ───────────────────────────────────────────
fig_dist_norm = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    subplot_titles=(
        "Driver Move Distance per VT Labor (km/labor)",
        "Driver Move as % of Total Distance",
    ),
    vertical_spacing=0.12,
)
for period_name, df, color in [
    (PERIOD_BEFORE, df_before, COLOR_BEFORE),
    (PERIOD_AFTER,  df_after,  COLOR_AFTER),
]:
    fig_dist_norm.add_trace(
        go.Scatter(x=df["date"], y=df["move_km_per_vt_labor"],
                   mode="lines+markers", name=period_name,
                   line=dict(color=color), marker=dict(size=4), legendgroup=period_name),
        row=1, col=1,
    )
    fig_dist_norm.add_trace(
        go.Scatter(x=df["date"], y=df["move_pct_of_total_distance"],
                   mode="lines+markers", name=period_name,
                   line=dict(color=color), marker=dict(size=4),
                   legendgroup=period_name, showlegend=False),
        row=2, col=1,
    )

if ADOPTION_DATE:
    for r in [1, 2]:
        fig_dist_norm.add_vline(x=ADOPTION_DATE, line_dash="dash", line_color="gray", row=r, col=1)

fig_dist_norm.update_layout(
    title="Normalized Distance KPIs Over Time",
    height=500,
    legend=dict(orientation="h", y=-0.15),
    template="plotly_white",
)
fig_dist_norm.show()

In [11]:
# ── Chart: Infeasibilities ─────────────────────────────────────────────────────
# Four distinct colors: infeasible before/after, overtime before/after
_COLOR_INFEAS_BEFORE   = "#EF553B"  # red
_COLOR_INFEAS_AFTER    = "#636EFA"  # blue
_COLOR_OVERTIME_BEFORE = "#FF9E3D"  # orange
_COLOR_OVERTIME_AFTER  = "#19D3F3"  # cyan

fig_infeas = go.Figure()
for period_name, df, color_infeas, color_overtime in [
    (PERIOD_BEFORE, df_before, _COLOR_INFEAS_BEFORE,  _COLOR_OVERTIME_BEFORE),
    (PERIOD_AFTER,  df_after,  _COLOR_INFEAS_AFTER,   _COLOR_OVERTIME_AFTER),
]:
    fig_infeas.add_trace(go.Scatter(
        x=df["date"], y=df["preassigned_infeasible"],
        mode="lines+markers", name=f"{period_name} — infeasible labors",
        line=dict(color=color_infeas), marker=dict(size=4),
    ))
    if "preassigned_overtime" in df.columns:
        fig_infeas.add_trace(go.Scatter(
            x=df["date"], y=df["preassigned_overtime"],
            mode="lines+markers", name=f"{period_name} — overtime",
            line=dict(color=color_overtime), marker=dict(size=4),
        ))

if ADOPTION_DATE:
    fig_infeas.add_vline(x=ADOPTION_DATE, line_dash="dash", line_color="gray",
                         annotation_text="Adoption", annotation_position="top right")

fig_infeas.update_layout(
    title="Infeasibilities & Overtime Over Time",
    xaxis_title="Date", yaxis_title="Count",
    legend=dict(orientation="h", y=-0.2),
    template="plotly_white",
)
fig_infeas.show()

In [12]:
# ── Chart: Drivers Used Per Day ────────────────────────────────────────────────
fig_drivers = go.Figure()
for period_name, df, color in [
    (PERIOD_BEFORE, df_before, COLOR_BEFORE),
    (PERIOD_AFTER,  df_after,  COLOR_AFTER),
]:
    fig_drivers.add_trace(go.Scatter(
        x=df["date"], y=df["drivers_used"],
        mode="lines+markers", name=period_name,
        line=dict(color=color), marker=dict(size=4),
    ))

if ADOPTION_DATE:
    fig_drivers.add_vline(x=ADOPTION_DATE, line_dash="dash", line_color="gray",
                          annotation_text="Adoption", annotation_position="top right")

fig_drivers.update_layout(
    title="Drivers Used Per Day",
    xaxis_title="Date", yaxis_title="Driver Count",
    legend=dict(orientation="h", y=-0.2),
    template="plotly_white",
)
fig_drivers.show()